## Find atomic contacts between antibody and antigen pairs

We want to create a DataFrame that contains atomic contacts for all the structures in our cleaned summary file.

The desired result is a DataFrame with columns

- `pdb_id`, e.g. '9ds1'
- `ab_chain`, e.g. 'H' 
- `ab_chaintype`, 'heavy' or 'light'
- `ag_resnum`
- `ab_resnumi`, including icode, e.g. '52A'
- `ab_resname`
- `ab_atom`
- `ag_chain`, e.g. 'G'
- `ag_resnum`
- `ag_resnumi`, e.g. '13'
- `ag_resname`, e.g. 'TYR'
- `ag_atom`



We can accomplish this using our `atomic_contact_points` function from notebook 07. Copy this function from the previous notebook into a code cell.

- read the cleaned summary file into a pandas DataFrame
- for each row of the summary DataFrame
    - read the PDB file
    - find atomic contact points for Hchain
    - find atomic contact points for Lchain
    - add columns pbs_id, ab_chaintype ('heavy' and 'light')

and concatenate to a single DataFrame.

Import required libraries

In [62]:
import os.path
from Bio.PDB import PDBParser, NeighborSearch
import pandas as pd

In [63]:
df = pd.read_csv("../data/ab_ag.tsv", sep="\t")

In [ ]:
DATASETS = [
    ("SARS-CoV-2", "../data/pdbs_corona_cnf"),   
    ("Influenza A", "../data/pdbs_influenza_cnf"),
    ("Homo Sapiens", "../data/pdbs_h.sapiens_cnf")
]

### Function - find atomic contact points

In [73]:
species = "Homo sapiens"  
unique_pdb_ids = df.query("species == @species")["pdb"].tolist()

UndefinedVariableError: name 'species' is not defined

In [66]:
def atomic_contact_points(ab_chain, ag_chain, distance):
    res = []
    ns = NeighborSearch(list(ag_chain.get_atoms()))
    for ab_atom in ab_chain.get_atoms():
        ab_res = ab_atom.get_parent()
        close_ag_atoms = ns.search(ab_atom.coord, distance)
        for ag_atom in close_ag_atoms:
            ag_res = ag_atom.get_parent()
            if ab_res.id[0] == ' ' and ag_res.id[0] == ' ':
                tmp = dict(ab_resnum = ab_res.id[1],
                           ab_icode = ab_res.id[2],
                           ab_resname = ab_res.get_resname(),
                           ab_atom = ab_atom.id,
                           ag_resnum = ag_res.id[1],
                           ag_icode = ag_res.id[2],
                           ag_resname = ag_res.get_resname(),
                           ag_atom = ag_atom.id)
                res.append(tmp)

    return pd.DataFrame(res)

In [ ]:
for species, pdb_dir in DATASETS:
    download_batch(species, pdb_dir)

### Function - find residue occurance

In [67]:
def residue_occurrence(chain):
    results = []
    for res in chain.get_residues():
        if res.id[0] == ' ' and res.id[1] <= 128:
            tmp = dict(ab_resnum = res.id[1],
                       ab_icode = res.id[2],
                       ab_resname = res.get_resname())
            results.append(tmp)

    return pd.DataFrame(results)

Define constants

In [ ]:
SUMMARY_FILE = '../generated/data cleanup/ab_ag_filtered_pdb.tsv'

PDB_DIR = '../data/pdbs_influenza_cnf'



In [ ]:
species = "Homo Sapiens"
df.query("species == @species")["pdb"].tolist()

now i try to make a loop so that all pdb directories can be run with the same code 

In [ ]:
for species, folder in species_to_dir.items():
    PDB_DIR = folder
    df_species = df.query("species == @species")
    print(f"\nAnalyzing {species}, using PDBs in: {PDB_DIR}")
    summary = pd.read_csv(SUMMARY_FILE, sep='\t')
summary.head()

NameError: name 'df' is not defined

In [ ]:
summary = pd.read_csv(SUMMARY_FILE, sep='\t')
summary.head()

,pdb,Hchain,Lchain,model,antigen_chain,antigen_type,antigen_name,compound,organism,heavy_species,light_species,antigen_species,resolution,method,scfv,engineered,heavy_subclass,light_subclass,light_ctype,species
0,8veb,G,I,0,E,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.97,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
1,8ved,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E11 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.98,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV2,Kappa,Influenza A
2,8vee,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 in compl...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,3.18,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
3,8vef,H,L,0,A,protein,hemagglutinin,Cryo-EM structure of antibody T5-1E08 UCA (unm...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,3.04,ELECTRON MICROSCOPY,False,True,IGHV4,IGKV1,Kappa,Influenza A
4,9dpc,H,L,0,D,protein,neuraminidase,Structure of Fab 297 in complex with influenza...,Homo sapiens; Influenza A virus,homo sapiens,homo sapiens,influenza a virus,2.65,ELECTRON MICROSCOPY,False,True,IGHV1,IGKV1,Kappa,Influenza A


In [ ]:
#import warnings
import sys
#from Bio.PDB.PDBExceptions import PDBConstructionWarning
#warnings.simplefilter("error", PDBConstructionWarning)


contacts = pd.DataFrame()
residues = pd.DataFrame()

for i, row in summary.iterrows():
    pdb_id = row['pdb']
    if pdb_id == '7mtb':
        continue
    hchain = row['Hchain']
    lchain = row['Lchain']
    antigen_chain = row['antigen_chain']

    try:
        if pd.isna(antigen_chain) or len(antigen_chain) > 1:
            continue

    
        filename = os.path.join(PDB_DIR, f'{pdb_id}')
        parser = PDBParser(PERMISSIVE=1)
        structure = parser.get_structure(pdb_id, filename)

        acph = atomic_contact_points(structure[0][hchain], structure[0][antigen_chain], 4.0)
        
        acph.insert(loc = 0, column = 'ag_chain', value = antigen_chain)
        acph.insert(loc = 0, column = 'ab_chain', value = hchain)
        acph.insert(loc = 0, column = 'chain_type', value = 'heavy')
        acph.insert(loc = 0, column = 'pdb_id', value = pdb_id)

        resh = residue_occurrence(structure[0][hchain])
        resh.insert(loc = 0, column = 'ab_chain', value = hchain)
        resh.insert(loc = 0, column = 'chain_type', value = 'heavy')
        resh.insert(loc = 0, column = 'pdb_id', value = pdb_id)


        acpl = atomic_contact_points(structure[0][lchain], structure[0][antigen_chain], 4.0)
        
        acpl.insert(loc = 0, column = 'ag_chain', value = antigen_chain)
        acpl.insert(loc = 0, column = 'ab_chain', value = lchain)
        acpl.insert(loc = 0, column = 'chain_type', value = 'light')
        acpl.insert(loc = 0, column = 'pdb_id', value = pdb_id)

        resl = residue_occurrence(structure[0][lchain])
        resl.insert(loc = 0, column = 'ab_chain', value = lchain)
        resl.insert(loc = 0, column = 'chain_type', value = 'light')
        resl.insert(loc = 0, column = 'pdb_id', value = pdb_id)

        residues = pd.concat([residues, resh, resl])
        contacts = pd.concat([contacts, acph, acpl])

    except Exception as e:
        print(e)
        print(row)
        sys.exit(1)




[Errno 2] No such file or directory: '../data/pdbs_influenza_cnf/8veb'
pdb                                                             8veb
Hchain                                                             G
Lchain                                                             I
model                                                              0
antigen_chain                                                      E
antigen_type                                                 protein
antigen_name                                           hemagglutinin
compound           Cryo-EM structure of antibody T5-1E08 in compl...
organism                             Homo sapiens; Influenza A virus
heavy_species                                           homo sapiens
light_species                                           homo sapiens
antigen_species                                    influenza a virus
resolution                                                      2.97
method                          

SystemExit: 1

/Users/loa/anaconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
contacts.to_csv("../generated/contacts/atomic_contacts.tsv", sep='\t', index = False)
contacts.head()

""


In [ ]:
residues.to_csv("../generated/contacts/residues.tsv", sep='\t', index = False)
residues.head()

,pdb_id,chain_type,ab_chain,ab_resnum,ab_icode,ab_resname
0,8veb,heavy,G,1,,GLN
1,8veb,heavy,G,2,,VAL
2,8veb,heavy,G,3,,GLN
3,8veb,heavy,G,4,,LEU
4,8veb,heavy,G,5,,LEU


In [ ]:
summary.pdb.nunique()


956